# vmChat — Colab 内置模型服务

使用 `google.colab.ai` 的当前账号模型与运行时凭据，无需手动填写 Gemini API Key。运行时不需要 GPU。

按顺序运行或点击“全部运行”。第三步会验证聊天、报告 DSL、鉴权、公网 SSE 与 CORS，并输出本地前端配置。

Colab 内置入口不支持原生 tool calling；适配层用文本传输工具请求，由现有 Python 工具执行资源读取与知识检索，保留服务的工具白名单与预算限制。

本地未推送的适配代码已内嵌，供直接运行。模型凭据仅放入进程环境，不写入笔记本输出或文件。

这是临时联调服务：运行时被回收后服务停止，重跑会生成新的公网地址和服务令牌。前端和内网业务接口仍在本地运行。

临时 trycloudflare 隧道不支持 SSE；前端请选择轮询进度，使用 start / snapshot / cancel 接口。当前账号额度不足时，两个 PDF 会明确返回 run.failed；health 成功不代表模型额度充足。


In [ ]:
# 1. 拉取当前分支，补充 Colab 适配层并安装依赖
import pathlib, subprocess, sys
repo=pathlib.Path('/content/vmchat-hermes-replacement-python')
branch='optimize/mandate-risk-generic-recall'
if not repo.exists():
    subprocess.run(['git','clone','--branch',branch,'--single-branch','https://github.com/liuruibing/vmchat-hermes-replacement-python.git',str(repo)],check=True)
actual_branch=subprocess.check_output(['git','branch','--show-current'],cwd=repo,text=True).strip()
assert actual_branch==branch, '现有 Colab 目录属于其他分支，请先检查目录'
print('代码提交',subprocess.check_output(['git','rev-parse','--short','HEAD'],cwd=repo,text=True).strip())
# 内嵌适配层使本次本地修改未推送时也能运行；仓库已有适配层时优先使用仓库版本。
adapter_path=repo/'app/provider/colab_provider.py'
if not adapter_path.exists():
    adapter_path.write_text('"""Text tool transport for Colab\'s built-in, text-only model proxy."""\n\nimport json\nimport os\nimport uuid\n\nfrom langchain_core.messages import AIMessageChunk, ToolMessage\nfrom langchain_core.utils.function_calling import convert_to_openai_tool\n\nfrom app.provider.langchain_provider import LangChainVmChatProvider, extract_first_json_value\n\n\nclass TextToolModel:\n    def __init__(self, model, tools=()):\n        self.model = model\n        self.tools = list(tools)\n\n    def __getattr__(self, name):\n        return getattr(self.model, name)\n\n    def bind_tools(self, tools):\n        return TextToolModel(self.model, tools)\n\n    async def astream(self, messages):\n        schemas = [convert_to_openai_tool(t)[\'function\'] for t in self.tools]\n        tool_names = {schema[\'name\'] for schema in schemas}\n        instruction = (\n            \'工具传输协议：此模型入口不支持原生工具调用。需要调用工具时，只输出一个 JSON 对象：\'\n            \'{"colab_tool_calls":[{"name":"工具名","arguments":{"参数名":"参数值"}}]}。\'\n            \'不要伪造工具结果。收到 TOOL_RESULT 后继续处理，可再次调用工具。\'\n            \'不需要工具或已取得资源时，直接按原任务要求输出最终答案或 DSL，\'\n            \'最终答案不要使用 colab_tool_calls 字段。可用工具：\\n\'\n            + json.dumps(schemas, ensure_ascii=False)\n        )\n        text_messages = [{\'role\': \'system\', \'content\': instruction}]\n        for message in messages:\n            if isinstance(message, dict):\n                text_messages.append(dict(message))\n            elif isinstance(message, ToolMessage):\n                text_messages.append({\n                    \'role\': \'user\',\n                    \'content\': \'TOOL_RESULT \' + message.tool_call_id + \'\\n\' + str(message.content),\n                })\n            elif getattr(message, \'tool_calls\', None):\n                text_messages.append({\n                    \'role\': \'assistant\',\n                    \'content\': json.dumps({\'colab_tool_calls\': [\n                        {\'name\': c[\'name\'], \'arguments\': c[\'args\'], \'id\': c[\'id\']}\n                        for c in message.tool_calls\n                    ]}, ensure_ascii=False),\n                })\n            else:\n                text_messages.append({\'role\': \'assistant\', \'content\': str(message.content)})\n\n        response = None\n        async for chunk in self.model.astream(text_messages):\n            response = response + chunk if response is not None else chunk\n        if response is None:\n            raise RuntimeError(\'COLAB_MODEL_EMPTY_RESPONSE\')\n        raw_json = extract_first_json_value(response.content) if isinstance(response.content, str) else None\n        try:\n            payload = json.loads(raw_json) if raw_json else None\n        except json.JSONDecodeError:\n            payload = None\n        if not isinstance(payload, dict) or \'colab_tool_calls\' not in payload:\n            yield response\n            return\n\n        calls = payload[\'colab_tool_calls\']\n        if not isinstance(calls, list) or not 1 <= len(calls) <= 8:\n            raise RuntimeError(\'COLAB_TOOL_REQUEST_INVALID: expected 1 to 8 tool calls\')\n        chunks = []\n        for index, call in enumerate(calls):\n            if (not isinstance(call, dict) or call.get(\'name\') not in tool_names\n                    or not isinstance(call.get(\'arguments\'), dict)):\n                raise RuntimeError(\'COLAB_TOOL_REQUEST_INVALID: unknown tool or invalid arguments\')\n            chunks.append({\n                \'name\': call[\'name\'], \'args\': json.dumps(call[\'arguments\'], ensure_ascii=False),\n                \'id\': \'colab_\' + uuid.uuid4().hex, \'index\': index,\n            })\n        yield AIMessageChunk(content=\'\', tool_call_chunks=chunks, usage_metadata=response.usage_metadata)\n\n\nclass ColabVmChatProvider(LangChainVmChatProvider):\n    def build_model(self, *, timeout_seconds=120):\n        model = super().build_model(timeout_seconds=timeout_seconds)\n        model.max_tokens = int(os.getenv(\'COLAB_MAX_OUTPUT_TOKENS\', \'16384\'))\n        return TextToolModel(model)\n',encoding='utf-8')

def patch_file(relative, old, new):
    path=repo/relative
    source=path.read_text()
    if new in source:return
    if old not in source:raise RuntimeError('源码已变化，需要核对适配入口：'+relative)
    path.write_text(source.replace(old,new),encoding='utf-8')

patch_file('app/config.py', 'llm_provider: Literal["langchain", "fixed"] = "fixed"', 'llm_provider: Literal["langchain", "fixed", "colab"] = "fixed"')
patch_file('app/config.py', '    llm_provider: Literal["langchain", "fixed"] = "langchain" if llm_provider_val == "langchain" else "fixed"', '    llm_provider: Literal["langchain", "fixed", "colab"] = (\n        llm_provider_val if llm_provider_val in ("langchain", "colab") else "fixed"\n    )')
patch_file('app/main.py', '    elif config.llm_provider == "langchain" and LangChainVmChatProvider is not None:\n        provider = LangChainVmChatProvider(', '    elif config.llm_provider in ("langchain", "colab") and LangChainVmChatProvider is not None:\n        provider_class = LangChainVmChatProvider\n        if config.llm_provider == "colab":\n            from app.provider.colab_provider import ColabVmChatProvider\n            provider_class = ColabVmChatProvider\n        provider = provider_class(')
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)

# 包含本次真实 PDF 联调发现的校验和进度传输修复。
(repo/'app/main.py').write_text('import asyncio\nfrom contextlib import asynccontextmanager\nfrom dataclasses import dataclass\nimport hmac\nimport json\nimport logging\nimport shutil\nimport tempfile\nfrom typing import Any, Dict, Optional\n\nfrom fastapi import FastAPI, Request\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import JSONResponse, StreamingResponse\n\nfrom app.config import AppConfig, load_config\nfrom app.compatibility.hermes_events import serialize_sse_event\nfrom app.compatibility.hermes_request import (\n    GlobalQueryParameters,\n    VmChatInput,\n    normalize_create_run_request,\n    normalize_vm_chat_input,\n)\nfrom app.contracts.types import VmChatRunError\nfrom app.resources.resource_loader import ResourceLoader\nfrom app.runs.run_limiter import RunLimiter\nfrom app.runs.run_store import RunStore\nfrom app.runs.run_types import AbortController\nfrom app.template_store import TemplateStore\n\nfrom app.provider.fixed_provider import FixedVmChatProvider, VmChatModelProvider\nfrom app.provider.langchain_provider import LangChainVmChatProvider\nfrom app.orchestrator.vmchat_orchestrator import stream_vm_chat\nfrom app.orchestrator.simple_chat_orchestrator import stream_simple_chat\nfrom app.workflow.engine import WorkflowContext, WorkflowEngine\nfrom app.workflow.legacy import build_default_workflow_registry\nfrom app.resources.wiki_catalog import get_wiki_tree, get_wiki_document\nfrom app.agents.registry import AgentRegistry\nfrom app.session.store import SqliteSessionStore\nfrom app.session.manager import SessionManager\nfrom app.context.manager import ContextManager\nfrom app.artifacts.store import SqliteArtifactStore\nfrom app.knowledge.store import DuckDbKnowledgeStore, SqliteKnowledgeStore\nfrom app.knowledge.service import KnowledgeService\nfrom app.knowledge.embeddings import OpenAICompatibleEmbeddingProvider\nfrom app.documents.multipart import parse_single_file_multipart\nfrom app.documents.parser import PdfDocumentParser\nfrom app.documents.service import DocumentService\nfrom app.documents.store import SqliteDocumentStore\n\nlogger = logging.getLogger("app")\n\n\nclass ActiveRunRegistry:\n    def __init__(self):\n        self._controllers: Dict[str, Any] = {}\n\n    def register(self, id_: str, controller: Any) -> None:\n        self._controllers[id_] = controller\n\n    def unregister(self, id_: str) -> None:\n        self._controllers.pop(id_, None)\n\n    def abort_all(self) -> None:\n        for controller in list(self._controllers.values()):\n            if hasattr(controller, "abort") and callable(controller.abort):\n                try:\n                    controller.abort()\n                except Exception:\n                    pass\n            elif hasattr(controller, "cancel") and callable(controller.cancel):\n                try:\n                    controller.cancel()\n                except Exception:\n                    pass\n        self._controllers.clear()\n\n    # Alias for JS camelCase\n    def abortAll(self) -> None:\n        self.abort_all()\n\n\nActiveRunRegistryImpl = ActiveRunRegistry\n\n\n@dataclass\nclass AppDependencies:\n    config: AppConfig\n    resource_loader: Any\n    run_store: Any\n    run_limiter: Any\n    provider: Any\n    active_registry: Optional[ActiveRunRegistry] = None\n    template_store: Any = None\n    agent_registry: Any = None\n    session_manager: Any = None\n    context_manager: Any = None\n    artifact_store: Any = None\n    knowledge_service: Any = None\n    document_service: Any = None\n    workflow_registry: Any = None\n    workflow_engine: Any = None\n\n    @property\n    def resourceLoader(self):\n        return self.resource_loader\n\n    @property\n    def runStore(self):\n        return self.run_store\n\n    @property\n    def runLimiter(self):\n        return self.run_limiter\n\n    @property\n    def activeRegistry(self):\n        return self.active_registry\n\n\ndef verify_bearer_auth(request: Request, api_key: str) -> bool:\n    if not api_key:\n        return True\n    if request.method == "OPTIONS":\n        return True\n\n    auth_header = request.headers.get("authorization") or request.headers.get("Authorization")\n    if not auth_header or not auth_header.startswith("Bearer "):\n        return False\n\n    token = auth_header[7:].strip()\n    token_bytes = token.encode("utf-8")\n    key_bytes = api_key.encode("utf-8")\n\n    return hmac.compare_digest(token_bytes, key_bytes)\n\n\n# Alias for JS camelCase compatibility\nverifyBearerAuth = verify_bearer_auth\n\n\ndef create_app(deps_override: Optional[Dict[str, Any]] = None) -> FastAPI:\n    if deps_override is None:\n        deps_override = {}\n\n    config = deps_override.get("config") or load_config()\n\n    agent_registry = (\n        deps_override.get("agent_registry")\n        or deps_override.get("agentRegistry")\n        or AgentRegistry(getattr(config, "agents_root", "agents")).load()\n    )\n    default_agent_id = getattr(config, "default_agent_id", "performance-ai")\n    default_role_id = getattr(config, "default_role_id", "performance-analyst")\n    default_agent = agent_registry.get(default_agent_id) if agent_registry else None\n\n    configured_resource_loader = deps_override.get("resource_loader") or deps_override.get("resourceLoader")\n    if configured_resource_loader is not None:\n        resource_loader = configured_resource_loader\n    elif default_agent is not None and default_agent.resourcesDir:\n        resource_loader = ResourceLoader(resources_dir=default_agent.resourcesDir)\n    else:\n        resource_loader = ResourceLoader()\n\n    run_store = deps_override.get("run_store") or deps_override.get("runStore") or RunStore(max_stored_runs=config.max_stored_runs)\n    run_limiter = deps_override.get("run_limiter") or deps_override.get("runLimiter") or RunLimiter()\n    active_registry = deps_override.get("active_registry") or deps_override.get("activeRegistry") or ActiveRunRegistry()\n    template_store = (\n        deps_override.get("template_store")\n        or deps_override.get("templateStore")\n        or TemplateStore(config.templates_file)\n    )\n\n    isolated_runtime = bool(deps_override)\n    default_session_path = ":memory:" if isolated_runtime else getattr(\n        config, "session_db_path", ".runtime/ai-sessions.sqlite3"\n    )\n    default_artifact_path = ":memory:" if isolated_runtime else getattr(\n        config, "artifact_db_path", ".runtime/ai-artifacts.sqlite3"\n    )\n    default_knowledge_path = ":memory:" if isolated_runtime else getattr(\n        config, "knowledge_db_path", ".runtime/knowledge.duckdb"\n    )\n    isolated_document_root = tempfile.mkdtemp(prefix="vmchat-documents-") if isolated_runtime else None\n    default_document_root = isolated_document_root or getattr(config, "document_root", ".runtime/documents")\n    default_document_db_path = ":memory:" if isolated_runtime else getattr(\n        config, "document_db_path", ".runtime/ai-documents.sqlite3"\n    )\n\n    session_store = (\n        deps_override.get("session_store")\n        or deps_override.get("sessionStore")\n        or SqliteSessionStore(default_session_path)\n    )\n    session_manager = (\n        deps_override.get("session_manager")\n        or deps_override.get("sessionManager")\n        or SessionManager(\n            session_store,\n            recent_messages=getattr(config, "session_recent_messages", 6),\n            relevant_history_messages=getattr(config, "session_relevant_history_messages", 4),\n        )\n    )\n    context_manager = (\n        deps_override.get("context_manager")\n        or deps_override.get("contextManager")\n        or ContextManager(\n            session_manager,\n            max_context_tokens=getattr(config, "max_context_tokens", 32000),\n            max_dsl_tokens=getattr(config, "max_context_dsl_tokens", 9000),\n            max_history_tokens=getattr(config, "max_context_history_tokens", 5000),\n            reserved_prompt_tokens=getattr(config, "max_context_reserved_tokens", 8000),\n        )\n    )\n    artifact_store = (\n        deps_override.get("artifact_store")\n        or deps_override.get("artifactStore")\n        or SqliteArtifactStore(default_artifact_path)\n    )\n\n    document_service = deps_override.get("document_service") or deps_override.get("documentService")\n    document_store = None\n    if document_service is None:\n        document_store = SqliteDocumentStore(default_document_db_path)\n        document_service = DocumentService(\n            root_dir=default_document_root,\n            store=document_store,\n            parser=PdfDocumentParser(),\n            max_document_bytes=getattr(config, "max_document_bytes", 20 * 1024 * 1024),\n        )\n\n    knowledge_service = deps_override.get("knowledge_service") or deps_override.get("knowledgeService")\n    knowledge_store = None\n    knowledge_backend = "custom" if knowledge_service is not None else "disabled"\n    embedding_provider = None\n    if knowledge_service is None:\n        if getattr(config, "embedding_model", "") and getattr(config, "embedding_api_key", ""):\n            try:\n                embedding_provider = OpenAICompatibleEmbeddingProvider(\n                    model=config.embedding_model,\n                    api_key=config.embedding_api_key,\n                    base_url=config.embedding_base_url or None,\n                )\n            except Exception as err:\n                logger.warning(f"[app] Embedding provider disabled: {err}")\n                embedding_provider = None\n\n        try:\n            knowledge_store = DuckDbKnowledgeStore(default_knowledge_path)\n            knowledge_service = KnowledgeService(knowledge_store, embedder=embedding_provider)\n            knowledge_backend = "duckdb"\n        except Exception as err:\n            logger.info(f"[app] DuckDB unavailable, using SQLite knowledge store: {err}")\n            fallback_path = str(default_knowledge_path)\n            if fallback_path.endswith(".duckdb"):\n                fallback_path = fallback_path[:-7] + ".sqlite3"\n            knowledge_store = SqliteKnowledgeStore(fallback_path)\n            knowledge_service = KnowledgeService(knowledge_store, embedder=embedding_provider)\n            knowledge_backend = "sqlite"\n\n    if deps_override.get("provider"):\n        provider = deps_override["provider"]\n    elif config.llm_provider in ("langchain", "colab") and LangChainVmChatProvider is not None:\n        provider_class = LangChainVmChatProvider\n        if config.llm_provider == "colab":\n            from app.provider.colab_provider import ColabVmChatProvider\n            provider_class = ColabVmChatProvider\n        provider = provider_class(\n            model_name=config.llm_model,\n            base_url=config.llm_base_url,\n            api_key=config.llm_api_key,\n            mode=config.llm_output_mode,\n        )\n    elif FixedVmChatProvider is not None:\n        provider = FixedVmChatProvider(config.fixed_provider_fixture)\n    else:\n        provider = None\n\n    workflow_registry = (\n        deps_override.get("workflow_registry")\n        or deps_override.get("workflowRegistry")\n        or build_default_workflow_registry(\n            simple_chat_stream=stream_simple_chat,\n            vm_report_stream=stream_vm_chat,\n        )\n    )\n    workflow_engine = (\n        deps_override.get("workflow_engine")\n        or deps_override.get("workflowEngine")\n        or WorkflowEngine(workflow_registry)\n    )\n\n    background_runs: dict[str, asyncio.Task] = {}\n\n    @asynccontextmanager\n    async def lifespan(app: FastAPI):\n        # Load resources at startup\n        try:\n            await resource_loader.load()\n            if knowledge_service is not None and resource_loader.is_ready():\n                loaded = resource_loader.get_resources()\n                resource_map = getattr(loaded, "toolResourceTextByPath", None) or getattr(\n                    loaded, "tool_resource_text_by_path", None\n                )\n                if isinstance(resource_map, dict):\n                    stats = await asyncio.to_thread(\n                        knowledge_service.index_resource_map,\n                        default_agent_id,\n                        resource_map,\n                    )\n                    logger.info(f"[app] Knowledge index ready for {default_agent_id}: {stats}")\n\n                if agent_registry is not None:\n                    for agent in agent_registry.list_agents():\n                        if agent.id == default_agent_id:\n                            continue\n                        generic_docs = agent_registry.collect_knowledge(agent.id)\n                        if generic_docs:\n                            stats = await asyncio.to_thread(\n                                knowledge_service.index_resource_map,\n                                agent.id,\n                                generic_docs,\n                            )\n                            logger.info(f"[app] Knowledge index ready for {agent.id}: {stats}")\n        except Exception as err:\n            logger.warning(f"[app] ResourceLoader/Knowledge load failed at startup: {err}")\n\n        yield\n\n        # Shutdown\n        active_registry.abort_all()\n        for task in background_runs.values():\n            task.cancel()\n        if background_runs:\n            await asyncio.gather(*background_runs.values(), return_exceptions=True)\n        if hasattr(run_store, "destroy") and callable(run_store.destroy):\n            run_store.destroy()\n        if hasattr(session_store, "close") and callable(session_store.close):\n            session_store.close()\n        if hasattr(artifact_store, "close") and callable(artifact_store.close):\n            artifact_store.close()\n        if document_store is not None and hasattr(document_store, "close"):\n            document_store.close()\n        if knowledge_store is not None and hasattr(knowledge_store, "close"):\n            knowledge_store.close()\n        if isolated_document_root:\n            shutil.rmtree(isolated_document_root, ignore_errors=True)\n\n    app = FastAPI(lifespan=lifespan)\n\n    app.add_middleware(\n        CORSMiddleware,\n        allow_origins=config.cors_origins,\n        allow_methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"],\n        allow_headers=["Content-Type", "Authorization"],\n    )\n\n    @app.middleware("http")\n    async def auth_and_degraded_middleware(request: Request, call_next):\n        is_run_or_wiki = request.url.path.startswith("/v1/runs") or request.url.path.startswith("/v1/wiki")\n        is_template = request.url.path.startswith("/v1/templates")\n        is_runtime = (\n            request.url.path.startswith("/v1/sessions")\n            or request.url.path.startswith("/v1/agents")\n            or request.url.path.startswith("/v1/knowledge")\n            or request.url.path.startswith("/v1/artifacts")\n            or request.url.path.startswith("/v1/documents")\n        )\n        if is_run_or_wiki or is_template or is_runtime:\n            if not verify_bearer_auth(request, config.service_api_key):\n                return JSONResponse(\n                    status_code=403,\n                    content={"error": "FORBIDDEN: Invalid or missing bearer token"}\n                )\n            if request.url.path.startswith("/v1/wiki") and not resource_loader.is_ready():\n                return JSONResponse(status_code=503, content={"error": "SERVICE_DEGRADED: Resources not loaded"})\n        return await call_next(request)\n\n    @app.get("/health")\n    async def health():\n        ready = resource_loader.is_ready()\n        status = "ok" if ready else "degraded"\n        status_code = 200 if ready else 503\n        model_configured = bool(config.llm_api_key or config.llm_provider == "fixed")\n\n        return JSONResponse(\n            status_code=status_code,\n            content={\n                "status": status,\n                "skillLoaded": ready,\n                "catalogLoaded": ready,\n                "schemaLoaded": ready,\n                "modelConfigured": model_configured,\n                "sessionStore": "sqlite",\n                "documentStore": "sqlite" if document_store is not None else "custom",\n                "maxDocumentBytes": getattr(config, "max_document_bytes", 20 * 1024 * 1024),\n                "knowledgeReady": knowledge_service is not None,\n                "knowledgeBackend": knowledge_backend,\n                "knowledgeVectorEnabled": embedding_provider is not None,\n                "defaultAgent": default_agent_id,\n                "workflowEngine": "registry",\n                "workflows": workflow_registry.list_ids() if workflow_registry is not None else [],\n                "langGraphEnabled": True,\n            }\n        )\n\n    @app.post("/v1/documents")\n    async def upload_document(request: Request):\n        max_document_bytes = getattr(config, "max_document_bytes", 20 * 1024 * 1024)\n        content_length = request.headers.get("content-length")\n        if content_length:\n            try:\n                if int(content_length) > max_document_bytes + 1024 * 1024:\n                    return JSONResponse(status_code=413, content={"error": "DOCUMENT_TOO_LARGE"})\n            except ValueError:\n                pass\n        try:\n            body = await request.body()\n            if len(body) > max_document_bytes + 1024 * 1024:\n                return JSONResponse(status_code=413, content={"error": "DOCUMENT_TOO_LARGE"})\n            filename, mime_type, content = parse_single_file_multipart(\n                request.headers.get("content-type") or "",\n                body,\n            )\n            record = await asyncio.to_thread(\n                document_service.create_pdf,\n                filename=filename,\n                mime_type=mime_type,\n                content=content,\n            )\n            return JSONResponse(status_code=201, content={"document": record.public_dict()})\n        except ValueError as err:\n            status_code = 413 if "DOCUMENT_TOO_LARGE" in str(err) else 400\n            return JSONResponse(status_code=status_code, content={"error": str(err)})\n        except Exception as err:\n            logger.exception("[documents] upload failed")\n            return JSONResponse(status_code=500, content={"error": f"DOCUMENT_UPLOAD_FAILED: {err}"})\n\n    @app.get("/v1/documents/{documentId}")\n    async def get_document(documentId: str):\n        record = document_service.get(documentId)\n        if record is None:\n            return JSONResponse(status_code=404, content={"error": "DOCUMENT_NOT_FOUND"})\n        return JSONResponse(status_code=200, content={"document": record.public_dict()})\n\n    @app.get("/v1/wiki/tree")\n    async def get_wiki_tree_endpoint():\n        if not resource_loader.is_ready():\n            return JSONResponse(\n                status_code=503,\n                content={"error": "SERVICE_DEGRADED: Resources not loaded"}\n            )\n        resources = resource_loader.get_resources()\n        return JSONResponse(status_code=200, content=get_wiki_tree(resources))\n\n    @app.get("/v1/wiki/documents/{documentId:path}")\n    async def get_wiki_document_endpoint(documentId: str):\n        if not resource_loader.is_ready():\n            return JSONResponse(\n                status_code=503,\n                content={"error": "SERVICE_DEGRADED: Resources not loaded"}\n            )\n        resources = resource_loader.get_resources()\n        doc = get_wiki_document(resources, documentId)\n        if doc is None:\n            return JSONResponse(\n                status_code=404,\n                content={"error": "WIKI_DOCUMENT_NOT_FOUND"}\n            )\n        return JSONResponse(status_code=200, content=doc)\n\n    @app.get("/v1/agents")\n    async def list_agents():\n        agents = []\n        for agent in agent_registry.list_agents() if agent_registry else []:\n            role = agent_registry.get_role(agent.id, agent.defaultRole) if agent_registry else None\n            agents.append({\n                "id": agent.id,\n                "name": agent.name,\n                "version": agent.version,\n                "workflow": agent.workflow,\n                "defaultRole": agent.defaultRole,\n                "defaultRoleName": role.name if role else None,\n                "metadata": agent.metadata,\n            })\n        return JSONResponse(status_code=200, content={"agents": agents})\n\n    @app.get("/v1/agents/{agentId}/roles")\n    async def list_agent_roles(agentId: str):\n        if agent_registry is None or agent_registry.get(agentId) is None:\n            return JSONResponse(status_code=404, content={"error": "AGENT_NOT_FOUND"})\n        roles = agent_registry.list_roles(agentId)\n        return JSONResponse(\n            status_code=200,\n            content={\n                "agent_id": agentId,\n                "roles": [\n                    {\n                        "id": role.id,\n                        "name": role.name,\n                        "allowedSkills": role.allowedSkills,\n                        "allowedTools": role.allowedTools,\n                        "knowledgeScopes": role.knowledgeScopes,\n                        "metadata": role.metadata,\n                    }\n                    for role in roles\n                ],\n            },\n        )\n\n    @app.get("/v1/sessions/{sessionId}")\n    async def get_session(sessionId: str):\n        state = session_store.get(sessionId)\n        if state is None:\n            return JSONResponse(status_code=404, content={"error": "SESSION_NOT_FOUND"})\n        payload = state.model_dump(by_alias=True)\n        payload["message_count"] = len(state.messages)\n        # Session inspection returns state, but not every historical message body.\n        payload["messages"] = [\n            {\n                "role": item.role,\n                "created_at": item.createdAt,\n                "run_id": item.runId,\n                "preview": item.content[:240],\n            }\n            for item in state.messages[-12:]\n        ]\n        return JSONResponse(status_code=200, content=payload)\n\n    @app.delete("/v1/sessions/{sessionId}")\n    async def delete_session(sessionId: str):\n        deleted = session_manager.delete(sessionId)\n        return JSONResponse(status_code=200, content={"ok": True, "deleted": deleted})\n\n    @app.post("/v1/knowledge/search")\n    async def search_knowledge(request: Request):\n        if knowledge_service is None:\n            return JSONResponse(status_code=503, content={"error": "KNOWLEDGE_SERVICE_UNAVAILABLE"})\n        body = await request.json()\n        query = str(body.get("query") or "").strip() if isinstance(body, dict) else ""\n        if not query:\n            return JSONResponse(status_code=400, content={"error": "query is required"})\n        agent_id = str(body.get("agent_id") or default_agent_id)\n        limit = min(20, max(1, int(body.get("limit") or 8)))\n        hits = knowledge_service.search(agent_id=agent_id, query=query, limit=limit)\n        return JSONResponse(\n            status_code=200,\n            content={\n                "query": query,\n                "agent_id": agent_id,\n                "hits": [\n                    {\n                        "path": hit.path,\n                        "title": hit.title,\n                        "content": hit.content,\n                        "score": hit.score,\n                        "metadata": hit.metadata,\n                    }\n                    for hit in hits\n                ],\n            },\n        )\n\n    @app.get("/v1/artifacts/{artifactId}")\n    async def get_artifact(artifactId: str):\n        artifact = artifact_store.get(artifactId)\n        if artifact is None:\n            return JSONResponse(status_code=404, content={"error": "ARTIFACT_NOT_FOUND"})\n        return JSONResponse(\n            status_code=200,\n            content=artifact.model_dump(by_alias=True),\n        )\n\n    @app.get("/v1/templates")\n    async def list_templates():\n        return JSONResponse(status_code=200, content={"ok": True, "templates": template_store.list()})\n\n    @app.get("/v1/templates/{templateId}")\n    async def get_template(templateId: str):\n        template = template_store.get(templateId)\n        if template is None:\n            return JSONResponse(status_code=404, content={"error": "TEMPLATE_NOT_FOUND"})\n        return JSONResponse(status_code=200, content={"ok": True, "template": template})\n\n    @app.put("/v1/templates")\n    async def replace_templates(request: Request):\n        try:\n            body = await request.json()\n            source = body.get("templates") if isinstance(body, dict) else body\n            if not isinstance(source, list):\n                raise ValueError("templates must be an array")\n            templates = template_store.replace_all(source)\n            return JSONResponse(status_code=200, content={"ok": True, "templates": templates})\n        except ValueError as err:\n            return JSONResponse(status_code=400, content={"error": f"INVALID_TEMPLATES: {err}"})\n        except Exception as err:\n            logger.exception("[templates] replace failed")\n            return JSONResponse(status_code=500, content={"error": f"SERVER_ERROR: {err}"})\n\n    @app.post("/v1/templates")\n    async def upsert_template(request: Request):\n        try:\n            body = await request.json()\n            template = body.get("template") if isinstance(body, dict) and isinstance(body.get("template"), dict) else body\n            saved = template_store.upsert(template)\n            return JSONResponse(status_code=200, content={"ok": True, "template": saved})\n        except ValueError as err:\n            return JSONResponse(status_code=400, content={"error": f"INVALID_TEMPLATE: {err}"})\n        except Exception as err:\n            logger.exception("[templates] upsert failed")\n            return JSONResponse(status_code=500, content={"error": f"SERVER_ERROR: {err}"})\n\n    @app.put("/v1/templates/{templateId}")\n    async def update_template(templateId: str, request: Request):\n        try:\n            body = await request.json()\n            template = body.get("template") if isinstance(body, dict) and isinstance(body.get("template"), dict) else body\n            saved = template_store.upsert(template, templateId)\n            return JSONResponse(status_code=200, content={"ok": True, "template": saved})\n        except ValueError as err:\n            return JSONResponse(status_code=400, content={"error": f"INVALID_TEMPLATE: {err}"})\n        except Exception as err:\n            logger.exception("[templates] update failed")\n            return JSONResponse(status_code=500, content={"error": f"SERVER_ERROR: {err}"})\n\n    @app.delete("/v1/templates/{templateId}")\n    async def delete_template(templateId: str):\n        if not template_store.delete(templateId):\n            return JSONResponse(status_code=404, content={"error": "TEMPLATE_NOT_FOUND"})\n        return JSONResponse(status_code=200, content={"ok": True, "deleted": templateId})\n\n    @app.post("/v1/runs")\n    async def create_run(request: Request):\n        try:\n            try:\n                body = await request.json()\n            except Exception:\n                body = {}\n            normalized_req = normalize_create_run_request(body)\n        except Exception as err:\n            return JSONResponse(\n                status_code=400,\n                content={"error": f"INVALID_REQUEST: {err}"}\n            )\n\n        try:\n            vm_input = normalize_vm_chat_input(normalized_req)\n\n            agent_id = normalized_req.agent_id or default_agent_id\n            role_id = normalized_req.role_id or default_role_id\n            agent = agent_registry.get(agent_id) if agent_registry else None\n            if agent_registry and agent is None:\n                raise ValueError(f"UNKNOWN_AGENT: {agent_id}")\n            if agent is not None and agent.workflow == "vm-report" and not resource_loader.is_ready():\n                raise RuntimeError("SERVICE_DEGRADED: Resources not loaded")\n            role = agent_registry.get_role(agent_id, role_id) if agent_registry else None\n            if agent is not None and role is None and agent.defaultRole:\n                role_id = agent.defaultRole\n                role = agent_registry.get_role(agent_id, role_id) if agent_registry else None\n\n            for document_id in normalized_req.documents:\n                document_service.require_ready(document_id)\n\n            if normalized_req.skills:\n                allowed_skills = set(role.allowedSkills if role is not None else [])\n                denied = [skill for skill in normalized_req.skills if skill not in allowed_skills]\n                if denied:\n                    raise ValueError(\n                        "SKILL_NOT_ALLOWED: " + ", ".join(denied)\n                    )\n\n            session_state = session_manager.get_or_create(\n                normalized_req.session_id,\n                agent_id=agent_id,\n                role_id=role_id,\n            )\n            prepared_input, context_report = context_manager.prepare(session_state, vm_input)\n            session_manager.update_ui_state(session_state, vm_input)\n        except Exception as err:\n            return JSONResponse(\n                status_code=400,\n                content={"error": f"INVALID_INPUT: {err}"}\n            )\n\n        try:\n            record = run_store.create(normalized_req, prepared_input)\n            req_obj = getattr(record, "request", None)\n            session_id = None\n            if isinstance(req_obj, dict):\n                session_id = req_obj.get("session_id")\n            elif hasattr(req_obj, "session_id"):\n                session_id = getattr(req_obj, "session_id")\n            if not session_id:\n                session_id = record.id\n\n            return JSONResponse(\n                status_code=200,\n                content={\n                    "run_id": record.id,\n                    "status": "queued",\n                    "session_id": session_id,\n                    "documents": list(normalized_req.documents),\n                    "context": {\n                        "estimated_tokens": context_report.total_estimated_tokens,\n                        "history_tokens": context_report.history_tokens,\n                        "dsl_tokens": context_report.dsl_tokens,\n                        "summary_tokens": context_report.summary_tokens,\n                        "budget_tokens": context_report.budget_tokens,\n                        "reserved_prompt_tokens": context_report.reserved_prompt_tokens,\n                        "dropped_history_messages": context_report.dropped_history_messages,\n                        "summary_truncated": context_report.summary_truncated,\n                        "full_dsl_blocks": context_report.full_dsl_blocks,\n                        "summarized_dsl_blocks": context_report.summarized_dsl_blocks,\n                    },\n                }\n            )\n        except Exception as err:\n            return JSONResponse(\n                status_code=500,\n                content={"error": f"SERVER_ERROR: {err}"}\n            )\n\n    @app.get("/v1/runs/{runId}/events")\n    async def run_events(runId: str, request: Request):\n        claim_result = run_store.claim(runId)\n        outcome = getattr(claim_result, "outcome", None) or (claim_result.get("outcome") if isinstance(claim_result, dict) else None)\n\n        if outcome == "not_found":\n            return JSONResponse(status_code=404, content={"error": "RUN_NOT_FOUND"})\n\n        if outcome == "streaming":\n            return JSONResponse(status_code=409, content={"error": "RUN_ALREADY_STREAMING"})\n\n        record = getattr(claim_result, "record", None) or (claim_result.get("record") if isinstance(claim_result, dict) else None)\n\n        if outcome == "replay":\n            async def replay_generator():\n                event_buffer = getattr(record, "eventBuffer", None) or getattr(record, "event_buffer", [])\n                for frame in event_buffer:\n                    yield frame\n\n            return StreamingResponse(\n                replay_generator(),\n                media_type="text/event-stream",\n                headers={\n                    "Cache-Control": "no-cache, no-transform",\n                    "Connection": "keep-alive",\n                    "X-Accel-Buffering": "no",\n                }\n            )\n\n        # outcome == "claimed"\n        try_acquire = getattr(run_limiter, "try_acquire", None) or getattr(run_limiter, "tryAcquire", None)\n        release_permit = try_acquire() if callable(try_acquire) else None\n\n        if not release_permit:\n            err_frame = serialize_sse_event({\n                "event": "run.failed",\n                "error": "系统繁忙，当前并发请求较多，请稍后重试",\n            })\n            run_store.fail(runId, err_frame)\n\n            async def limit_err_generator():\n                yield err_frame\n\n            return StreamingResponse(\n                limit_err_generator(),\n                media_type="text/event-stream",\n                headers={\n                    "Cache-Control": "no-cache, no-transform",\n                    "Connection": "keep-alive",\n                    "X-Accel-Buffering": "no",\n                }\n            )\n\n        controller = getattr(record, "controller", None) or AbortController()\n        if active_registry:\n            active_registry.register(runId, controller)\n\n        async def event_generator():\n            is_normal_ended = False\n\n            def is_aborted() -> bool:\n                if hasattr(controller, "is_aborted") and controller.is_aborted:\n                    return True\n                signal = getattr(controller, "signal", None)\n                if signal and getattr(signal, "aborted", False):\n                    return True\n                return False\n\n            def write_sse_frame(frame: str, allow_aborted: bool = False) -> str:\n                if not allow_aborted and is_aborted():\n                    raise RuntimeError("CLIENT_DISCONNECTED: Client disconnected during streaming")\n                run_store.append_event(runId, frame)\n                return frame\n\n            try:\n                vm_input = (\n                    getattr(record, "normalizedInput", None)\n                    or getattr(record, "normalized_input", None)\n                    or VmChatInput(\n                        userMessage="Default prompt",\n                        selectedBlockId=None,\n                        currentDsls=[],\n                        globalQueryParameters=GlobalQueryParameters(names=[], nonemptyFlags={}),\n                        historyMessages=[],\n                    )\n                )\n\n                agent_id = getattr(vm_input, "agentId", None) or default_agent_id\n                role_id = getattr(vm_input, "roleId", None) or default_role_id\n                agent = agent_registry.get(agent_id) if agent_registry else None\n                if agent_registry is not None and agent is None:\n                    raise VmChatRunError("UNKNOWN_AGENT", f"未知 Agent: {agent_id}")\n                workflow = agent.workflow if agent is not None else "vm-report"\n                role = agent_registry.get_role(agent_id, role_id) if agent_registry else None\n                role_prompt = role.systemPrompt if role else ""\n                knowledge_search = (\n                    (lambda query: knowledge_service.format_for_tool(agent_id, query))\n                    if knowledge_service is not None\n                    else None\n                )\n\n                full_output = ""\n                has_reasoning = False\n                sequence = 0\n                last_usage: Dict[str, Any] = {}\n                completion_metadata = None\n\n                skill_md = agent_registry.read_skill(agent_id) if agent_registry else ""\n                loaded_resources = (\n                    resource_loader.get_resources()\n                    if resource_loader.is_ready()\n                    else None\n                )\n                document_ids = list(getattr(vm_input, "documentIds", []) or [])\n                workflow_context = WorkflowContext(\n                    input_val=vm_input,\n                    provider=provider,\n                    agent_id=agent_id,\n                    role_id=role_id,\n                    role_prompt=role_prompt,\n                    skill_md=skill_md,\n                    knowledge_search=knowledge_search,\n                    resources=loaded_resources,\n                    signal=controller,\n                    message_chunk_chars=config.sse_chunk_chars,\n                    run_id=runId,\n                    session_id=getattr(getattr(record, "request", None), "session_id", None),\n                    document_ids=document_ids,\n                    document_loader=document_service.load_parsed if document_ids else None,\n                )\n                try:\n                    stream_gen = workflow_engine.stream(\n                        workflow,\n                        workflow_context,\n                    ).__aiter__()\n                except ValueError as err:\n                    raise VmChatRunError(\n                        "UNKNOWN_WORKFLOW",\n                        str(err),\n                    ) from err\n\n                if stream_gen is not None:\n                    next_event_task = asyncio.create_task(stream_gen.__anext__())\n\n                    try:\n                        while True:\n                            done, _ = await asyncio.wait({next_event_task}, timeout=12.0)\n                            if not done:\n                                if not is_aborted():\n                                    yield ": keep-alive\\n\\n"\n                                continue\n\n                            try:\n                                event = next_event_task.result()\n                            except StopAsyncIteration:\n                                break\n\n                            next_event_task = asyncio.create_task(stream_gen.__anext__())\n\n                            if is_aborted():\n                                raise RuntimeError("CLIENT_DISCONNECTED")\n\n                            event_type = getattr(event, "event", None) or getattr(event, "type", None) or (\n                                (event.get("event") or event.get("type")) if isinstance(event, dict) else None\n                            )\n\n                            if event_type == "reasoning.delta":\n                                has_reasoning = True\n                                sequence += 1\n                                delta = getattr(event, "delta", None) or (event.get("delta") if isinstance(event, dict) else "")\n                                frame = write_sse_frame(serialize_sse_event({\n                                    "event": "reasoning.delta",\n                                    "delta": delta,\n                                    "sequence": sequence,\n                                }))\n                                yield frame\n                            elif event_type == "message.delta":\n                                delta = getattr(event, "delta", None) or (event.get("delta") if isinstance(event, dict) else "")\n                                full_output += delta\n                                frame = write_sse_frame(serialize_sse_event({\n                                    "event": "message.delta",\n                                    "delta": delta,\n                                }))\n                                yield frame\n                            elif event_type == "stream.done":\n                                event_output = getattr(event, "output", None) or (event.get("output") if isinstance(event, dict) else "")\n                                if event_output != full_output:\n                                    raise VmChatRunError(\n                                        "STREAM_OUTPUT_MISMATCH",\n                                        "AI 助手输出校验失败，请重试",\n                                    )\n                                last_usage = getattr(event, "usage", None) or (event.get("usage") if isinstance(event, dict) else {}) or {}\n                            elif event_type == "run.completed":\n                                event_output = getattr(event, "output", None) or (event.get("output") if isinstance(event, dict) else "")\n                                full_output = event_output or full_output\n                                last_usage = getattr(event, "usage", None) or (event.get("usage") if isinstance(event, dict) else {}) or {}\n                                completion_metadata = getattr(event, "metadata", None) if not isinstance(event, dict) else event.get("metadata")\n                            elif event_type == "run.failed":\n                                fail_msg = getattr(event, "error", None) or (event.get("error") if isinstance(event, dict) else "") or "AI 助手任务执行失败"\n                                err_frame = serialize_sse_event({"event": "run.failed", "error": fail_msg})\n                                run_store.fail(runId, err_frame)\n                                yield err_frame\n                                return\n                    finally:\n                        if not next_event_task.done():\n                            next_event_task.cancel()\n                        await asyncio.gather(next_event_task, return_exceptions=True)\n\n                if has_reasoning:\n                    frame = write_sse_frame(serialize_sse_event({"event": "reasoning.done"}))\n                    yield frame\n\n                completion_event = {\n                    "event": "run.completed",\n                    "output": full_output,\n                    "usage": last_usage,\n                }\n                if completion_metadata is not None:\n                    completion_event["metadata"] = completion_metadata\n                frame = write_sse_frame(serialize_sse_event(completion_event))\n                yield frame\n\n                artifact_id = None\n                try:\n                    parsed_output = json.loads(full_output)\n                    is_dsl_artifact = isinstance(parsed_output, list) or (\n                        isinstance(parsed_output, dict)\n                        and any(key in parsed_output for key in ("action", "requests", "view", "views"))\n                    )\n                    if is_dsl_artifact:\n                        artifact = artifact_store.create(\n                            session_id=getattr(getattr(record, "request", None), "session_id", None) or runId,\n                            kind="vm-report-dsl",\n                            payload=parsed_output,\n                            metadata={"runId": runId},\n                        )\n                        artifact_id = artifact.id\n                except Exception:\n                    artifact_id = None\n\n                request_obj = getattr(record, "request", None)\n                session_id_for_turn = (\n                    getattr(request_obj, "session_id", None)\n                    if request_obj is not None\n                    else None\n                ) or runId\n                session_state = session_manager.get_or_create(\n                    session_id_for_turn,\n                    agent_id=getattr(vm_input, "agentId", default_agent_id),\n                    role_id=getattr(vm_input, "roleId", default_role_id),\n                )\n                session_manager.complete_turn(\n                    session_state,\n                    vm_input.userMessage,\n                    full_output,\n                    run_id=runId,\n                    artifact_id=artifact_id,\n                )\n\n                is_normal_ended = True\n                run_store.complete(runId, full_output)\n\n            except (Exception, asyncio.CancelledError, GeneratorExit) as err:\n                msg = str(err) or err.__class__.__name__\n                if not is_normal_ended:\n                    if not (isinstance(err, (asyncio.CancelledError, GeneratorExit)) or "CLIENT_DISCONNECTED" in msg):\n                        import traceback\n                        traceback.print_exc()\n                    fail_msg = (\n                        "客户端连接已中断"\n                        if isinstance(err, (asyncio.CancelledError, GeneratorExit)) or "CLIENT_DISCONNECTED" in msg\n                        else getattr(err, "public_message", None) if isinstance(err, VmChatRunError)\n                        else getattr(err, "publicMessage", None) if isinstance(err, VmChatRunError)\n                        else "AI 助手任务执行失败"\n                    )\n                    err_frame = serialize_sse_event({"event": "run.failed", "error": fail_msg})\n                    try:\n                        write_sse_frame(err_frame, allow_aborted=True)\n                    except Exception:\n                        pass\n                    run_store.fail(runId, err_frame)\n                    yield err_frame\n            finally:\n                if active_registry:\n                    active_registry.unregister(runId)\n                if callable(release_permit):\n                    release_permit()\n\n        return StreamingResponse(\n            event_generator(),\n            media_type="text/event-stream",\n            headers={\n                "Cache-Control": "no-cache, no-transform",\n                "Connection": "keep-alive",\n                "X-Accel-Buffering": "no",\n            }\n        )\n\n    @app.post("/v1/runs/{runId}/start")\n    async def start_background_run(runId: str, request: Request):\n        record = run_store.get(runId)\n        if record is None:\n            return JSONResponse(status_code=404, content={"error": "RUN_NOT_FOUND"})\n        if record.status != "queued":\n            return JSONResponse(status_code=200, content={"run_id": runId, "status": record.status})\n\n        response = await run_events(runId, request)\n        if not isinstance(response, StreamingResponse):\n            return response\n\n        async def consume():\n            async for frame in response.body_iterator:\n                if isinstance(frame, bytes):\n                    frame = frame.decode("utf-8")\n                if frame.startswith("data:") and (\n                    not record.eventBuffer or record.eventBuffer[-1] != frame\n                ):\n                    run_store.append_event(runId, frame)\n\n        task = asyncio.create_task(consume())\n        background_runs[runId] = task\n        def finish_task(done):\n            background_runs.pop(runId, None)\n            error = None if done.cancelled() else done.exception()\n            if record.status in ("running", "validating", "repairing") and (done.cancelled() or error):\n                frame = serialize_sse_event({\n                    "event": "run.failed",\n                    "error": "客户端连接已中断" if done.cancelled() else "后台任务执行失败",\n                })\n                run_store.append_event(runId, frame)\n                run_store.fail(runId, frame)\n\n        task.add_done_callback(finish_task)\n        return JSONResponse(status_code=202, content={"run_id": runId, "status": record.status})\n\n    @app.post("/v1/runs/{runId}/cancel")\n    async def cancel_background_run(runId: str):\n        record = run_store.get(runId)\n        if record is None:\n            return JSONResponse(status_code=404, content={"error": "RUN_NOT_FOUND"})\n        if record.status == "queued":\n            frame = serialize_sse_event({"event": "run.failed", "error": "客户端连接已中断"})\n            run_store.append_event(runId, frame)\n            run_store.fail(runId, frame)\n        elif record.status in ("running", "validating", "repairing"):\n            if record.controller:\n                record.controller.abort()\n            task = background_runs.get(runId)\n            if task is not None:\n                task.cancel()\n        return JSONResponse(content={"run_id": runId, "status": record.status})\n\n    @app.get("/v1/runs/{runId}/snapshot")\n    async def run_snapshot(runId: str, after: int = 0):\n        record = run_store.get(runId)\n        if record is None:\n            return JSONResponse(status_code=404, content={"error": "RUN_NOT_FOUND"})\n        frames = list(record.eventBuffer)\n        if after < 0 or after > len(frames):\n            return JSONResponse(status_code=400, content={"error": "INVALID_EVENT_CURSOR"})\n        batch = frames[after:after + 100]\n        return JSONResponse(content={\n            "run_id": runId, "status": record.status,\n            "cursor": after + len(batch), "frames": batch,\n        }, headers={"Cache-Control": "no-store"})\n\n    return app\n\n\n# Default app instance\napp = create_app()\n\n# Aliases for JS compatibility\nbuildApp = create_app\nbuild_app = create_app\n',encoding='utf-8')
(repo/'app/config.py').write_text('import os\nimport pathlib\nfrom typing import Dict, List, Literal, Optional\nfrom dotenv import load_dotenv\nfrom pydantic import BaseModel, Field\n\n\nload_dotenv()\n\n\nclass AppConfig(BaseModel):\n    host: str = "127.0.0.1"\n    port: int = 7310\n    service_api_key: str = ""\n    cors_origins: List[str] = Field(default_factory=lambda: ["http://localhost:9528", "http://127.0.0.1:9528"])\n    llm_provider: Literal["langchain", "fixed", "colab"] = "fixed"\n    llm_base_url: str = ""\n    llm_api_key: str = ""\n    llm_model: str = "deepseek-v4-flash"\n    llm_output_mode: Literal["structured", "text-json"] = "structured"\n    llm_network_retries: int = 0\n    allow_missing_usage: bool = False\n    max_prompt_chars: int = 120000\n    max_stored_runs: int = 100\n    max_final_output_chars: int = 200000\n    fixed_provider_fixture: str = ""\n    request_body_limit_bytes: int = 1048576\n    sse_chunk_chars: int = 256\n    vmchat_frontend_root: str = ""\n    templates_file: str = ".runtime/vmchat-templates.json"\n    default_agent_id: str = "performance-ai"\n    default_role_id: str = "performance-analyst"\n    agents_root: str = "agents"\n    session_db_path: str = ".runtime/ai-sessions.sqlite3"\n    artifact_db_path: str = ".runtime/ai-artifacts.sqlite3"\n    knowledge_db_path: str = ".runtime/knowledge.duckdb"\n    document_root: str = ".runtime/documents"\n    document_db_path: str = ".runtime/ai-documents.sqlite3"\n    max_document_bytes: int = 20 * 1024 * 1024\n    session_recent_messages: int = 6\n    session_relevant_history_messages: int = 4\n    max_context_tokens: int = 32000\n    max_context_dsl_tokens: int = 9000\n    max_context_history_tokens: int = 5000\n    max_context_reserved_tokens: int = 8000\n    embedding_model: str = ""\n    embedding_base_url: str = ""\n    embedding_api_key: str = ""\n\n    # JS camelCase property aliases\n    @property\n    def serviceApiKey(self) -> str:\n        return self.service_api_key\n\n    @property\n    def corsOrigins(self) -> List[str]:\n        return self.cors_origins\n\n    @property\n    def llmProvider(self) -> str:\n        return self.llm_provider\n\n    @property\n    def llmBaseUrl(self) -> str:\n        return self.llm_base_url\n\n    @property\n    def llmApiKey(self) -> str:\n        return self.llm_api_key\n\n    @property\n    def llmModel(self) -> str:\n        return self.llm_model\n\n    @property\n    def llmOutputMode(self) -> str:\n        return self.llm_output_mode\n\n    @property\n    def llmNetworkRetries(self) -> int:\n        return self.llm_network_retries\n\n    @property\n    def allowMissingUsage(self) -> bool:\n        return self.allow_missing_usage\n\n    @property\n    def maxPromptChars(self) -> int:\n        return self.max_prompt_chars\n\n    @property\n    def maxStoredRuns(self) -> int:\n        return self.max_stored_runs\n\n    @property\n    def maxFinalOutputChars(self) -> int:\n        return self.max_final_output_chars\n\n    @property\n    def fixedProviderFixture(self) -> str:\n        return self.fixed_provider_fixture\n\n    @property\n    def requestBodyLimitBytes(self) -> int:\n        return self.request_body_limit_bytes\n\n    @property\n    def sseChunkChars(self) -> int:\n        return self.sse_chunk_chars\n\n    @property\n    def vmchatFrontendRoot(self) -> str:\n        return self.vmchat_frontend_root\n\n    @property\n    def templatesFile(self) -> str:\n        return self.templates_file\n\n\ndef load_config(env: Optional[Dict[str, Optional[str]]] = None) -> AppConfig:\n    if env is None:\n        raw_env = dict(os.environ)\n    else:\n        raw_env = {k: v for k, v in env.items() if v is not None}\n\n    llm_retries_str = raw_env.get("LLM_NETWORK_RETRIES", "0")\n    try:\n        llm_network_retries = int(llm_retries_str)\n    except ValueError:\n        llm_network_retries = 0\n\n    if llm_network_retries != 0:\n        raise ValueError("INVALID_CONFIG: LLM_NETWORK_RETRIES must be 0 because LLM generation POST is not idempotent")\n\n    cors_raw = raw_env.get("CORS_ORIGINS", "http://localhost:9528,http://127.0.0.1:9528")\n    cors_origins = [s.strip() for s in cors_raw.split(",") if s.strip()]\n\n    frontend_root_raw = raw_env.get("VMCHAT_FRONTEND_ROOT", "../jn-fof-gf-vue")\n    vmchat_frontend_root = str(pathlib.Path.cwd().joinpath(frontend_root_raw).resolve())\n\n    llm_provider_val = raw_env.get("LLM_PROVIDER", "fixed")\n    llm_provider: Literal["langchain", "fixed", "colab"] = (\n        llm_provider_val if llm_provider_val in ("langchain", "colab") else "fixed"\n    )\n\n    llm_output_mode_val = raw_env.get("LLM_OUTPUT_MODE", "structured")\n    llm_output_mode: Literal["structured", "text-json"] = "text-json" if llm_output_mode_val == "text-json" else "structured"\n\n    host = raw_env.get("HOST", "127.0.0.1")\n    port = int(raw_env.get("PORT", "7310"))\n    service_api_key = raw_env.get("SERVICE_API_KEY", "")\n    llm_base_url = raw_env.get("LLM_BASE_URL", "")\n    llm_api_key = raw_env.get("LLM_API_KEY") or raw_env.get("OPENAI_API_KEY", "")\n    llm_model = raw_env.get("LLM_MODEL", "deepseek-v4-flash")\n    allow_missing_usage = raw_env.get("ALLOW_MISSING_USAGE", "").lower() == "true"\n    max_prompt_chars = int(raw_env.get("MAX_PROMPT_CHARS", "120000"))\n    max_stored_runs = int(raw_env.get("MAX_STORED_RUNS", "100"))\n    max_final_output_chars = int(raw_env.get("MAX_FINAL_OUTPUT_CHARS", "200000"))\n    fixed_provider_fixture = raw_env.get("FIXED_PROVIDER_FIXTURE", "")\n    request_body_limit_bytes = int(raw_env.get("REQUEST_BODY_LIMIT_BYTES", "1048576"))\n    sse_chunk_chars = int(raw_env.get("SSE_CHUNK_CHARS", "256"))\n    templates_file = raw_env.get("VMCHAT_TEMPLATES_FILE", ".runtime/vmchat-templates.json")\n    default_agent_id = raw_env.get("DEFAULT_AGENT_ID", "performance-ai")\n    default_role_id = raw_env.get("DEFAULT_ROLE_ID", "performance-analyst")\n    agents_root = raw_env.get("AGENTS_ROOT", "agents")\n    session_db_path = raw_env.get("SESSION_DB_PATH", ".runtime/ai-sessions.sqlite3")\n    artifact_db_path = raw_env.get("ARTIFACT_DB_PATH", ".runtime/ai-artifacts.sqlite3")\n    knowledge_db_path = raw_env.get("KNOWLEDGE_DB_PATH", ".runtime/knowledge.duckdb")\n    document_root = raw_env.get("DOCUMENT_ROOT", ".runtime/documents")\n    document_db_path = raw_env.get("DOCUMENT_DB_PATH", ".runtime/ai-documents.sqlite3")\n    max_document_bytes = int(raw_env.get("MAX_DOCUMENT_BYTES", str(20 * 1024 * 1024)))\n    session_recent_messages = int(raw_env.get("SESSION_RECENT_MESSAGES", "6"))\n    session_relevant_history_messages = int(raw_env.get("SESSION_RELEVANT_HISTORY_MESSAGES", "4"))\n    max_context_tokens = int(raw_env.get("MAX_CONTEXT_TOKENS", "32000"))\n    max_context_dsl_tokens = int(raw_env.get("MAX_CONTEXT_DSL_TOKENS", "9000"))\n    max_context_history_tokens = int(raw_env.get("MAX_CONTEXT_HISTORY_TOKENS", "5000"))\n    max_context_reserved_tokens = int(raw_env.get("MAX_CONTEXT_RESERVED_TOKENS", "8000"))\n    embedding_model = raw_env.get("EMBEDDING_MODEL", "")\n    embedding_base_url = raw_env.get("EMBEDDING_BASE_URL", "")\n    embedding_api_key = raw_env.get("EMBEDDING_API_KEY", "")\n\n    return AppConfig(\n        host=host,\n        port=port,\n        service_api_key=service_api_key,\n        cors_origins=cors_origins,\n        llm_provider=llm_provider,\n        llm_base_url=llm_base_url,\n        llm_api_key=llm_api_key,\n        llm_model=llm_model,\n        llm_output_mode=llm_output_mode,\n        llm_network_retries=0,\n        allow_missing_usage=allow_missing_usage,\n        max_prompt_chars=max_prompt_chars,\n        max_stored_runs=max_stored_runs,\n        max_final_output_chars=max_final_output_chars,\n        fixed_provider_fixture=fixed_provider_fixture,\n        request_body_limit_bytes=request_body_limit_bytes,\n        sse_chunk_chars=sse_chunk_chars,\n        vmchat_frontend_root=vmchat_frontend_root,\n        templates_file=templates_file,\n        default_agent_id=default_agent_id,\n        default_role_id=default_role_id,\n        agents_root=agents_root,\n        session_db_path=session_db_path,\n        artifact_db_path=artifact_db_path,\n        knowledge_db_path=knowledge_db_path,\n        document_root=document_root,\n        document_db_path=document_db_path,\n        max_document_bytes=max_document_bytes,\n        session_recent_messages=session_recent_messages,\n        session_relevant_history_messages=session_relevant_history_messages,\n        max_context_tokens=max_context_tokens,\n        max_context_dsl_tokens=max_context_dsl_tokens,\n        max_context_history_tokens=max_context_history_tokens,\n        max_context_reserved_tokens=max_context_reserved_tokens,\n        embedding_model=embedding_model,\n        embedding_base_url=embedding_base_url,\n        embedding_api_key=embedding_api_key,\n    )\n\n\n# Alias for JS camelCase compatibility\nloadConfig = load_config\n',encoding='utf-8')
(repo/'app/mandate_risk_v2/constraint_validator.py').write_text('from __future__ import annotations\n\nimport json\nimport re\nfrom decimal import Decimal, InvalidOperation\nfrom typing import Any, Iterable\n\nfrom app.mandate_risk.clauses import DocumentClause\nfrom app.mandate_risk_v2.models import RequirementConstraint, RequirementDraft\n\n\n_COMPARATIVE_OPERATORS = {\n    "<", "<=", ">", ">=", "=", "==",\n    "LT", "LTE", "GT", "GTE", "EQ",\n}\n_BETWEEN_OPERATORS = {"BETWEEN", "RANGE"}\n\n\ndef _normalize_text(value: str) -> str:\n    text = re.sub(r"(?<=\\w)-[ \\t]*\\r?\\n[ \\t]*(?=\\w)", "-", str(value or ""))\n    return re.sub(r"\\s+", " ", text).strip().casefold()\n\n\ndef _scalar_text(value: Any) -> str | None:\n    if value is None or isinstance(value, bool) or isinstance(value, (dict, list, tuple, set)):\n        return None\n    if isinstance(value, float) and value.is_integer():\n        return str(int(value))\n    return str(value).strip()\n\n\ndef _numeric(value: Any) -> Decimal | None:\n    if isinstance(value, bool) or value is None:\n        return None\n    try:\n        return Decimal(str(value))\n    except (InvalidOperation, ValueError, TypeError):\n        return None\n\n\ndef _evidence_text(requirement: RequirementDraft, clauses_by_id: dict[str, DocumentClause]) -> str:\n    return " ".join(\n        clauses_by_id[clause_id].text\n        for clause_id in requirement.evidence.clause_ids\n        if clause_id in clauses_by_id\n    )\n\n\ndef _validate_constraint_shape(constraint: RequirementConstraint, *, owner: str) -> None:\n    operator = str(constraint.operator or "").strip().upper()\n    if operator in _COMPARATIVE_OPERATORS and constraint.value is None:\n        if not (constraint.benchmark or "").strip() and not (constraint.formula or "").strip():\n            raise ValueError(f"{owner} comparative constraint requires value, benchmark or formula")\n        if not constraint.raw_value_text:\n            raise ValueError(f"{owner} symbolic comparison requires raw_value_text")\n    if operator in _BETWEEN_OPERATORS:\n        if constraint.value is None or constraint.value_to is None:\n            raise ValueError(f"{owner} range constraint requires value and value_to")\n        left = _numeric(constraint.value)\n        right = _numeric(constraint.value_to)\n        if left is not None and right is not None and right < left:\n            raise ValueError(f"{owner} range constraint has value_to < value")\n\n    if (constraint.value is not None or constraint.value_to is not None) and not constraint.raw_value_text:\n        raise ValueError(f"{owner} quantitative constraint requires raw_value_text")\n\n\ndef _validate_constraint_provenance(\n    constraint: RequirementConstraint,\n    *,\n    evidence_text: str,\n    owner: str,\n) -> None:\n    normalized_evidence = _normalize_text(evidence_text)\n    raw_value_text = _normalize_text(constraint.raw_value_text or "")\n    if raw_value_text and raw_value_text not in normalized_evidence:\n        raise ValueError(f"{owner} raw_value_text is not present in evidence")\n    if constraint.value is None and str(constraint.operator or "").strip().upper() in _COMPARATIVE_OPERATORS:\n        benchmark = _normalize_text(constraint.benchmark or "")\n        if benchmark and benchmark not in normalized_evidence:\n            raise ValueError(f"{owner} benchmark is not present in evidence")\n\n    # Verify scalar values are grounded in the copied raw value span. This is\n    # deliberately lexical only: Python does not infer financial semantics,\n    # rating order, percentage validity, or unit conversions.\n    source_for_values = raw_value_text or normalized_evidence\n    for label, value in (("value", constraint.value), ("value_to", constraint.value_to)):\n        token = _scalar_text(value)\n        if token is None:\n            continue\n        normalized_token = _normalize_text(token)\n        if normalized_token and normalized_token not in source_for_values:\n            raise ValueError(f"{owner} {label} is not present in raw evidence text")\n\n\ndef validate_requirement_constraints(\n    requirements: Iterable[RequirementDraft],\n    *,\n    clauses: Iterable[DocumentClause],\n) -> None:\n    """Fail closed on structurally or textually ungrounded constraint fields.\n\n    This validator never decides what a clause *means*. It only verifies that\n    model-supplied normalized values are internally complete and traceable to\n    the Python-owned canonical evidence selected by that Requirement.\n    """\n\n    clauses_by_id = {clause.clause_id: clause for clause in clauses}\n    for requirement in requirements:\n        constraint = requirement.constraint\n        if constraint is None:\n            continue\n        owner = f"requirement {requirement.local_id}"\n        evidence_text = _evidence_text(requirement, clauses_by_id)\n        try:\n            _validate_constraint_shape(constraint, owner=owner)\n            _validate_constraint_provenance(constraint, evidence_text=evidence_text, owner=owner)\n        except ValueError as exc:\n            context = json.dumps({\n                "constraint": constraint.model_dump(exclude_none=True),\n                "evidence_text": evidence_text,\n            }, ensure_ascii=False, separators=(",", ":"))\n            raise ValueError(f"{exc}; original constraint and evidence: {context}") from exc\n',encoding='utf-8')
(repo/'app/mandate_risk_v2/mapping_prompts.py').write_text('from __future__ import annotations\n\nimport json\nfrom typing import Sequence\n\nfrom app.mandate_risk.clauses import DocumentClause\nfrom app.mandate_risk.models import RawRiskMetric\nfrom app.mandate_risk_v2.mapping_models import CORE_COMPATIBILITY_DIMENSIONS, critic_targets\nfrom app.mandate_risk_v2.models import RequirementIR\n\n_CORE_DIMENSIONS_TEXT = ", ".join(CORE_COMPATIBILITY_DIMENSIONS)\n\nMAPPING_SYSTEM_PROMPT = f"""你是 Mandate PDF 的风险指标筛选员。任务是筛选符合文档要求、目标、策略或风险暴露的原始库指标，供用户判断；不是确认自动入库、合规通过或算法完全等价。\n只用 Requirement IR、逐字原文及给定原始库行，不创造指标、阈值或实际组合数值。\n完整浏览分配给你的每个库行；有合理监控用途的指标应保留。合同没有点名指标、库算法为空、年化/事前口径不明，不是自动排除理由，写出差异即可。\n每个相关行至少保留一条最有依据的 Requirement link；只有多个独立要求才增加关联，不输出笛卡尔积。仅凭一般风险政策或共同关键词，不能推出整个库都有用；无相关资产暴露的专属指标也不能牵强解释为排除性监控。\nlevel 的含义：DIRECT 是与 PDF 明确测量概念或要求直接对应；REVIEW 是有合理的间接监控关联或适用性仍需用户判断；REJECTED 是无合理用途。DIRECT 不代表算法等价或入库确认。\n每个 DIRECT/REVIEW 必须返回 match_score（0–100 的数字）和 score_reason（具体评分理由）。这是模型评估的文档匹配程度，不是统计校准的正确概率、文本向量余弦相似度或用户认可率。\n评分参考：90–100 为文档有明确对应要求且对象、资产/策略适用性充分支持；75–89 为业务关联清晰，存在需要说明的口径差异；50–74 为合理但较间接的监控用途或适用条件依据较弱；0–49 为关联弱或有重大不适用因素。独立依据原文评分，不按指标名称、链接类型、固定分数或凑数量评分；不确定性须反映在理由中。Python 不按分数阈值自动过滤。\ncompatibility 只解释实际影响匹配的维度，不要求填齐十二项。可参考：{_CORE_DIMENSIONS_TEXT}，也可增加合同特有维度；至少解释一个实质关联或差异。\n每项写 requirement_basis、metric_basis、relation 和 reason；relation 只能是 EQUIVALENT、INSUFFICIENT、CONFLICT、NOT_APPLICABLE。算法缺失不能猜等价；计价基础、时点、年化、事前/事后或资产适用性差异要写清，但不因差异自动丢掉有用途的指标。\nrow_assessments.outcome 只能 LINKED 或 NOT_RELEVANT；本批有该行的任意 link 则 LINKED，否则 NOT_RELEVANT。DIRECT、REVIEW、REJECTED 只能用于 link.level。\n每个 link 的 evidence_clause_ids 至少包含该 Requirement 自身 evidence.clause_ids 中的一项；定义、上下文和有 IR relation 的其他 Requirement 条款可补充主证据，不发明 clause_id。\n只输出一个 JSON 对象。"""\n\nDESTINATION_SYSTEM_PROMPT = """你是 Mandate 筛选结果覆盖审计员。\n逐条说明每个 Requirement 的独立对象、时间窗口和条件分支。去向只能 MAIN_TABLE、PENDING_REVIEW、LIBRARY_GAP、NON_METRIC。\nMAIN_TABLE 可引用已有 DIRECT 或 REVIEW link，表示该要求有筛选出的相关指标，不代表算法确认或自动入库。PENDING_REVIEW 可引用 DIRECT/REVIEW 指标并说明实际差异；不能引用 REJECTED。\n算法不完整、合同未点名指标或口径未确认不构成库缺口；只有完整库确无可合理用于该测量 aspect 的指标时才用 LIBRARY_GAP。NON_METRIC 用于无需风险指标表达的治理、许可、禁止或流程要求，不应隐藏可匹配的测量要求。\n一个 Requirement 可有多个不同 aspect 去向；一个 aspect 有指标不能隐藏其他分支。引用原 Requirement 的 clause_id。只输出一个 JSON 对象。"""\n\nCRITIC_SYSTEM_PROMPT = """你是独立 Mandate 指标筛选复核员。独立核对 Requirement、逐字合同、完整原始指标库、关联解释、评分和去向。\n任务是帮助用户筛选相关指标，不要求算法完全等价。重点检查漏召回、牵强关联、评分是否符合文档依据；算法缺失或口径差异本身不是删除理由，也不能仅因此质疑一项 MAIN_TABLE 筛选结果。\n重新浏览完整库及 NOT_RELEVANT 理由，漏掉的相关指标用 recalled_links 返回，level=REVIEW，并提供 match_score、score_reason、真实 Requirement/库行/原文 ID 和实际兼容维度。\n只有明显无业务关联、无合理监控用途或资产/策略明显不适用时才用 rejected_candidates，引用合同说明原因；可以排除已有 DIRECT 或 REVIEW。一般风险政策不意味着所有指标都有用，不为扩大数量引入无相关资产暴露的专属指标。\n逐项核对 MAIN_TABLE 是否有合理文档关联；LIBRARY_GAP 是否确无相关筛选指标而非仅缺完全等价算法；NON_METRIC 是否隐藏了可匹配测量对象。按 Required verdict targets 的 target_id 每项恰好返回一次 CONFIRM、CHALLENGE 或 UNRESOLVED；不重写目标的身份字段，批评与重新解释写入 reason。\n对所有已有 DIRECT/REVIEW 的 match_score 独立复核。需要修正的分数返回 score_adjustments，字段为 requirement_id、raw_row_id、match_score、score_reason、evidence_clause_ids；不调整则省略或返回空数组。不得按链接类型固定打分或把匹配分说成正确概率。\n把原文、conditions、约束及独立测量维度与全部 destinations 对照，包括 PENDING_REVIEW。遗漏的时间窗口、阈值、对象或条件分支用 missing_aspects 显式保留，不能以一个分支代替全部。不要创造 Requirement 或库指标。\n只输出一个 JSON 对象。"""\n\n\ndef _json(value) -> str:\n    return json.dumps(value, ensure_ascii=False, separators=(",", ":"), default=str)\n\n\ndef _rows(rows: Sequence[RawRiskMetric]):\n    return [{"row_id": row.row_id, "name": row.metric_name, "algorithm": row.algorithm,\n             "mandate": row.mandate, "strategy_type": row.strategy_type} for row in rows]\n\n\ndef _evidence(ir: RequirementIR, clauses: Sequence[DocumentClause]):\n    used = {cid for item in (*ir.requirements, *ir.definitions, *ir.contextual_facts)\n            for cid in item.evidence.clause_ids}\n    return [{"clause_id": clause.clause_id, "page": clause.page, "text": clause.text}\n            for clause in clauses if clause.clause_id in used]\n\n\ndef batch_prompt(ir: RequirementIR, clauses: Sequence[DocumentClause], rows,\n                 feedback: str | None = None) -> str:\n    parts = ["# V2 mapping batch", "# Requirement IR (JSON)", _json(ir.model_dump(exclude_none=True, exclude_defaults=True)),\n             "# Canonical evidence (JSON)", _json(_evidence(ir, clauses)),\n             "# Raw metric rows (JSON)", _json(_rows(rows)),\n             "# Optional compatibility reference dimensions", _json(list(CORE_COMPATIBILITY_DIMENSIONS)),\n             \'compatibility 项 JSON 形状必须为 {"dimension":"measurement_object","requirement_basis":"合同原文支持的口径","metric_basis":"库算法支持的口径","relation":"EQUIVALENT","reason":"双方依据"}；relation 只能取 EQUIVALENT / INSUFFICIENT / CONFLICT / NOT_APPLICABLE。只写实际相关维度；不要求填齐十二项，口径差异作为说明而非筛选门禁。\',\n             \'row_assessments 项形状为 {"raw_row_id":真实库行ID,"outcome":"LINKED","reason":"已有本行link"} 或 {"raw_row_id":真实库行ID,"outcome":"NOT_RELEVANT","reason":"无关联link的原因"}；outcome 只允许 LINKED / NOT_RELEVANT，不能使用 link.level 的值。\',\n             "返回 {links:[{requirement_id,raw_row_id,level,match_score,score_reason,compatibility:[{dimension,requirement_basis,metric_basis,relation,reason}],evidence_clause_ids,reason}],row_assessments:[{raw_row_id,outcome,reason}]}。每个库行必须恰好一个 row_assessment。"]\n    parts.append("身份与证据检查：对每个 link，先按 requirement_id 找到 IR 中的对应 Requirement，复制它自身 evidence.clause_ids 作为主证据；不要因指标名称或合同条款相似而交换 Requirement 编号。补充证据仅限定义、上下文或该 Requirement 的 IR relation 所关联的条款。输出前逐项核对。")\n    if feedback:\n        parts.extend(["# Previous invalid output feedback", feedback])\n    return "\\n".join(parts)\n\n\ndef destinations_prompt(ir: RequirementIR, clauses: Sequence[DocumentClause], rows,\n                        links, feedback: str | None = None) -> str:\n    parts = ["# V2 final destinations", "# Requirement IR (JSON)", _json(ir.model_dump(exclude_none=True, exclude_defaults=True)),\n             "# Canonical evidence (JSON)", _json(_evidence(ir, clauses)),\n             "# Complete raw metric catalogue (JSON)", _json(_rows(rows)),\n             "# Validated screening links (JSON)", _json([\n                 {**link.model_dump(exclude={"compatibility", "score_reason"}, exclude_none=True),\n                  "differences": [{"dimension": item.dimension, "relation": item.relation,\n                                   "reason": item.reason} for item in link.compatibility\n                                  if item.relation in {"INSUFFICIENT", "CONFLICT"}]}\n                 for link in links\n             ]),\n             "返回 {dispositions:[{requirement_id,reason,destinations:[{destination,raw_row_ids,evidence_clause_ids,aspect,reason}]}]}。",\n             "逐项列出每个 Requirement 的独立时间窗口、阈值、测量对象和条件分支；一个维度有候选不代表其他维度已有去向。每个独立 aspect 应各有去向，或在同一去向中完整说明全部维度及差异。"]\n    if feedback:\n        parts.extend(["# Previous review feedback", feedback])\n    return "\\n".join(parts)\n\n\ndef critic_prompt(ir: RequirementIR, clauses: Sequence[DocumentClause], rows,\n                  links, dispositions, feedback: str | None = None, row_assessments=()) -> str:\n    parts = [\n        "# V2 independent critic", "# Requirement IR (JSON)", _json(ir.model_dump(exclude_none=True, exclude_defaults=True)),\n        "# Canonical evidence (JSON)", _json(_evidence(ir, clauses)),\n        "# Complete raw metric catalogue (JSON)", _json(_rows(rows)),\n        "# Validated mapping links (JSON)", _json([x.model_dump() for x in links]),\n        "# Proposed destinations (JSON)", _json(dispositions.model_dump()),\n        "# Required verdict targets (JSON)", _json(critic_targets(dispositions)),\n        "# Full catalogue row assessments (JSON)", _json([x.model_dump() for x in row_assessments]),\n        \'返回 {verdicts:[{"target_id":"TGT-0001","verdict":"CONFIRM","reason":"独立核对结论","evidence_clause_ids":["真实原文ID"]}]}。Required verdict targets 中每个 target_id 恰好返回一次，不遗漏、不重复、不创造编号；无需返回 requirement_id、destination、raw_row_id、aspect，Python 按编号保留这些身份字段。PENDING_REVIEW 不在 verdict targets 内，遗漏的独立维度仍通过 missing_aspects 报告。\',\n        "另返回 recalled_links:[] 与 rejected_candidates:[]。recalled_links 使用与映射 links 相同形状，level 只能 REVIEW；rejected_candidates 项为 {requirement_id,raw_row_id,reason,evidence_clause_ids}，可引用已有 DIRECT/REVIEW。无遗漏/误报时返回空数组。",\n        "另返回 score_adjustments:[]，项形状为 {requirement_id,raw_row_id,match_score,score_reason,evidence_clause_ids}；只调整已有且未排除的 DIRECT/REVIEW，分数为0–100的数字。recalled_links 也必须有 match_score 和 score_reason。",\n        "另返回 missing_aspects:[]，逐条列出已抽取但 destinations 遗漏的独立维度，形状为 {requirement_id,aspect,reason,evidence_clause_ids}；引用真实 Requirement 自身证据，aspect 与该 Requirement 已有 destination 不重复。没有遗漏时返回空数组。",\n    ]\n    if feedback:\n        parts.extend(["# Previous critic validation feedback", feedback])\n    return "\\n".join(parts)\n',encoding='utf-8')
subprocess.run(['uv','sync','--locked','--no-dev','--no-editable'],cwd=repo,check=True)


In [ ]:
# 2. 自动使用 Colab 自带模型，不需要 Gemini API Key
from google.colab import ai
import os, secrets, subprocess, pathlib, requests, time
models=ai.list_models()
MODEL_ID='google/gemini-2.5-flash-lite' if 'google/gemini-2.5-flash-lite' in models else 'google/gemini-2.5-flash'
# 内置库已经在首次成功请求后缓存运行时模型凭据，仅传入子进程，不打印或保存。
proxy_token=os.environ.get('MODEL_PROXY_API_KEY')
assert proxy_token, 'Colab 模型凭据未初始化，先运行内置模型验证单元'
proxy_host=os.environ['MODEL_PROXY_HOST'].rstrip('/')
service_key=globals().get('service_key') or 'vmchat_'+secrets.token_urlsafe(24)
os.environ.update({'LLM_PROVIDER':'colab','LLM_BASE_URL':proxy_host+'/models/openapi','LLM_API_KEY':proxy_token,'LLM_MODEL':MODEL_ID,'LLM_OUTPUT_MODE':'text-json','LLM_NETWORK_RETRIES':'0','SERVICE_API_KEY':service_key,'CORS_ORIGINS':'http://localhost:9528,http://127.0.0.1:9528','VMCHAT_SQL_KNOWLEDGE_PATH':'delivery/02_vm_modules_sql_statements.md','MAX_SKILL_RESOURCE_READS':'24','MAX_PROMPT_CHARS':'120000','SESSION_DB_PATH':'/content/ai-sessions.sqlite3','ARTIFACT_DB_PATH':'/content/ai-artifacts.sqlite3','KNOWLEDGE_DB_PATH':'/content/knowledge.sqlite3'})
for name in ('vmchat_uvicorn','vmchat_cloudflared'):
    proc=globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate();proc.wait(timeout=10)
log_path=pathlib.Path('/content/vmchat-uvicorn.log')
vmchat_uvicorn=subprocess.Popen(['uv','run','uvicorn','app.main:app','--host','0.0.0.0','--port','8000'],cwd=repo,env=os.environ.copy(),stdout=open(log_path,'w'),stderr=subprocess.STDOUT)
for _ in range(60):
    if vmchat_uvicorn.poll() is not None:
        raise RuntimeError(log_path.read_text(errors='ignore')[-6000:])
    try:
        response=requests.get('http://127.0.0.1:8000/health',timeout=2)
        if response.status_code==200:
            print('HEALTH',response.status_code,response.json());break
    except requests.RequestException: pass
    time.sleep(1)
else: raise RuntimeError(log_path.read_text(errors='ignore')[-6000:])
print('VMCHAT_BUILTIN_MODEL',MODEL_ID)
print('SERVICE_READY')


In [ ]:
# 验证真实模型请求、服务鉴权、SSE，再开放临时公网地址
import json, requests, pathlib, subprocess, time, os, re, urllib.request
base='http://127.0.0.1:8000'
headers={'Authorization':'Bearer '+service_key}
unauthorized=requests.post(base+'/v1/runs',json={},timeout=10)
print('UNAUTHORIZED',unauthorized.status_code)
assert unauthorized.status_code in (401,403), unauthorized.text

def check_run(base, agent, role, prompt):
    body={'model':MODEL_ID,'input':[{'role':'user','content':prompt}],'instructions':'','session_id':'colab-check-'+os.urandom(6).hex(),'agent_id':agent,'role_id':role,'skills':[],'tools':[],'context':{}}
    created=requests.post(base+'/v1/runs',json=body,headers=headers,timeout=30)
    print('CREATE_RUN',agent,created.status_code)
    created.raise_for_status()
    run_id=created.json()['run_id']
    with requests.get(base+'/v1/runs/'+run_id+'/events',headers=headers,stream=True,timeout=180) as stream:
        stream.raise_for_status()
        for line in stream.iter_lines():
            if not line.startswith(b'data: '):continue
            event=json.loads(line[6:])
            if event.get('event') in ('run.completed','run.failed'):
                print('RUN_RESULT',agent,json.dumps(event,ensure_ascii=False)[:2200])
                assert event['event']=='run.completed',event
                assert event.get('output'),event
                return event
    raise RuntimeError('缺少终态 SSE 事件')
chat_result=check_run(base,'mandate-risk-chat','risk-assistant','请用一句话说明你可以帮助我什么。')
report_result=check_run(base,'performance-ai','performance-analyst','创建一个股票净敞口时序折线图。')
cloudflared=pathlib.Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cloudflared)
    cloudflared.chmod(0o755)
tunnel_log_path=pathlib.Path('/content/cloudflared.log')
vmchat_cloudflared=subprocess.Popen([str(cloudflared),'tunnel','--url',base,'--no-autoupdate'],stdout=open(tunnel_log_path,'w'),stderr=subprocess.STDOUT)
for _ in range(60):
    text=tunnel_log_path.read_text(errors='ignore')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
    if match:
        public_url=match.group(0);break
    if vmchat_cloudflared.poll() is not None:raise RuntimeError(text[-3000:])
    time.sleep(1)
else:raise RuntimeError(text[-3000:])
for _ in range(30):
    try:
        health=requests.get(public_url+'/health',timeout=10)
        if health.status_code==200:break
    except requests.RequestException:pass
    time.sleep(2)
else:raise RuntimeError('公网健康检查未通过')
print('PUBLIC_URL',public_url)
print('PUBLIC_HEALTH',health.status_code,health.json())
preflight=requests.options(public_url+'/v1/runs',headers={'Origin':'http://localhost:9528','Access-Control-Request-Method':'POST','Access-Control-Request-Headers':'authorization,content-type'},timeout=15)
print('CORS_PREFLIGHT',preflight.status_code,preflight.headers.get('access-control-allow-origin'))
assert preflight.status_code==200
assert preflight.headers.get('access-control-allow-origin')=='http://localhost:9528'
public_result=check_run(public_url,'mandate-risk-chat','risk-assistant','只用中文一句话回复：Colab 内置模型已连接。')
print('FRONTEND_CONFIG')
print("localStorage.setItem('fof-research-hermes-base-url', "+json.dumps(public_url)+");")
print("localStorage.setItem('fof-research-hermes-api-key', "+json.dumps(service_key)+");")
print("localStorage.removeItem('fof-research-hermes-run-events-disabled');")
print('location.reload();')
print('ALL_CHECKS_PASSED')


In [ ]:
# 默认不停止，确保“全部运行”后服务继续可用。
STOP_SERVICES = False # @param {type:"boolean"}
if STOP_SERVICES:
    for name in ('vmchat_cloudflared','vmchat_uvicorn'):
        proc=globals().get(name)
        if proc and proc.poll() is None:proc.terminate()
    print('服务已停止')
